# Module B3 : Qiskit — exécution sur QPU

Ce notebook montre comment exécuter des circuits quantiques sur différentes **backends**
(simulateurs et QPUs réels).

**Plan :**
1. Différentes backends (simulateurs et QPUs réels)
2. Transpilation
3. Primitives (Sampler, Estimator)
4. Démo avec une paire de Bell

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

---
## 1. Différentes backends

- **Simulateurs** : exécution sur ordinateur classique (ex : `AerSimulator`).
  Utiles pour le développement et le débogage.
- **QPUs réels** : processeurs quantiques accessibles via IBM Quantum
  (ex : `ibm_brisbane`). Introduisent du bruit (erreurs de portes, décohérence).

In [ ]:
# Simulateur local avec Aer (si disponible)
try:
    from qiskit_aer import AerSimulator
    simulator = AerSimulator()
    print("Backend simulateur :", simulator.name)
except ImportError:
    print("qiskit_aer non installé. On utilisera le Sampler de qiskit.primitives.")
    simulator = None

Pour accéder à un **QPU réel**, il faut un compte IBM Quantum et configurer
le service `QiskitRuntimeService` :

```python
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService(channel='ibm_quantum')
backend = service.least_busy(operational=True, simulator=False)
print(backend.name)
```

---
## 2. Transpilation

Un circuit écrit avec des portes abstraites (H, CNOT, …) doit être
**transpilé** pour correspondre aux contraintes physiques du QPU cible.

### Les 6 étapes de la transpilation

1. **Init** : décomposition des opérations multi-qubits en portes à 1 et 2 qubits
2. **Layout** : assignation des qubits logiques aux qubits physiques
3. **Routing** : insertion de portes SWAP pour les qubits non-adjacents (NP-difficile !)
4. **Translation** : conversion vers le jeu d'instructions natif (ISA) du QPU
5. **Optimization** : remplacement ou élimination de portes redondantes
6. **Scheduling** : insertion de délais (optionnel)

### Niveaux d'optimisation

| Niveau | Description |
|--------|-------------|
| 1 | Combinaison matricielle des portes 1-qubit, élimination des CNOT redondants |
| 2 | Analyse de commutation pour éliminer davantage de redondances |
| 3 | Optimisation des sous-blocs 2-qubits (reconstruction d'unitaires équivalents) |

In [ ]:
# Exemple de transpilation sur simulateur
bell_circuit = QuantumCircuit(2)
bell_circuit.h(0)
bell_circuit.cx(0, 1)
bell_circuit.measure_all()

print("Circuit original :")
display(bell_circuit.draw('mpl'))

if simulator is not None:
    transpiled = transpile(bell_circuit, backend=simulator, optimization_level=1)
    print("\nCircuit transpilé (niveau 1) :")
    display(transpiled.draw('mpl'))

---
## 3. Primitives

Les **primitives** sont l'interface standard pour exécuter des circuits dans Qiskit :

- **`Sampler`** : exécute le circuit et retourne un échantillon de résultats de mesure.
- **`Estimator`** : estime la valeur moyenne d'une observable.

Elles sont disponibles pour les simulateurs et les QPUs réels.

In [ ]:
# Utiliser le StatevectorSampler local (sans backend réel)
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

# Exécuter le circuit de Bell avec 1000 shots
job = sampler.run([bell_circuit], shots=1000)
result = job.result()

# Extraire les résultats
pub_result = result[0]
counts = pub_result.data.meas.get_counts()
print("Résultats (1000 shots) :", counts)
plot_histogram(counts)

On observe uniquement $\ket{00}$ et $\ket{11}$ avec des fréquences proches de 50%.

Sur un **vrai QPU**, on observerait aussi un peu de bruit : quelques
$\ket{01}$ et $\ket{10}$ apparaîtraient dans les résultats.

---
## 4. Démo complète : paire de Bell sur backend

On résume les étapes :
1. Construire le circuit
2. Transpiler pour le backend cible
3. Exécuter avec le Sampler
4. Analyser les résultats

In [ ]:
# 1. Construire le circuit de Bell
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()
print("Circuit :")
display(qc.draw('mpl'))

# 2. Transpiler (ici pour le simulateur)
if simulator is not None:
    qc_transpiled = transpile(qc, backend=simulator, optimization_level=2)
    print("\nCircuit transpilé :")
    display(qc_transpiled.draw('mpl'))
else:
    qc_transpiled = qc

# 3. Exécuter
sampler = StatevectorSampler()
job = sampler.run([qc_transpiled], shots=1000)
result = job.result()

# 4. Analyser
counts = result[0].data.meas.get_counts()
print("\nRésultats :")
plot_histogram(counts)

### Exercice

<mark>**Exercice B3.1**</mark> : Construire un circuit GHZ à 3 qubits
($\ket{\text{GHZ}} = \frac{1}{\sqrt{2}}(\ket{000} + \ket{111})$),
le transpiler et l'exécuter avec 2000 shots. Quels résultats attendez-vous ?